# Preparation

First, we'll define a function that we will use when building our agent.

It will generate fake weather data:

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

In [11]:
import json

In [8]:
import random

known_weather_data = {
    'berlin': 20.0
}

def get_weather(city: str) -> float:
    city = city.strip().lower()

    if city in known_weather_data:
        return known_weather_data[city]

    return round(random.uniform(-5, 35), 1)

### Q1. Define function description

We want to use it as a tool for our agent, so we need to describe it

How should the description for this function look like? Fill in missing parts

In [2]:
"""
get_weather_tool = {
    "type": "function",
    "name": "<TODO1>",
    "description": "<TODO2>",
    "parameters": {
        "type": "object",
        "properties": {
            "<TODO3>": {
                "type": "string",
                "description": "<TODO4>"
            }
        },
        "required": [TODO5],
        "additionalProperties": False
    }
}
"""

'\nget_weather_tool = {\n    "type": "function",\n    "name": "<TODO1>",\n    "description": "<TODO2>",\n    "parameters": {\n        "type": "object",\n        "properties": {\n            "<TODO3>": {\n                "type": "string",\n                "description": "<TODO4>"\n            }\n        },\n        "required": [TODO5],\n        "additionalProperties": False\n    }\n}\n'

In [25]:
get_weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get weather function",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "Provide city text to look up weather in that city"
            }
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

### Testing it (Optional)

If you have OpenAI API Key (or alternative provider), let's test it.

A question could be "What's the weather like in Germany?"

Experiment with different system prompts to have better answers from the system.

You can use chat_assistant.py or implement everything yourself

In [26]:
import chat_assistant

tools = chat_assistant.Tools()
tools.add_tool(get_weather, get_weather_tool)

In [27]:
tools.get_tools()

[{'type': 'function',
  'name': 'get_weather',
  'description': 'Get weather function',
  'parameters': {'type': 'object',
   'properties': {'city': {'type': 'string',
     'description': 'Provide city text to look up weather in that city'}},
   'required': ['city'],
   'additionalProperties': False}}]

In [28]:
question = "What's the weather like in Germany?"

developer_prompt = """
You're a weather assistant. 
You're given a question about weather in some city and your task is to answer it.
""".strip()

chat_interface = chat_assistant.ChatInterface()

chat = chat_assistant.ChatAssistant(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    client=client
)

In [29]:
chat.run()

Chat ended.


### Q2. Adding another tool

Let's add another tool - a function that can add weather data to our database:

In [50]:
def set_weather(city: str, temp: float) -> None:
    city = city.strip().lower()
    known_weather_data[city] = temp
    return 'OK'

In [49]:
"""
def set_weather(city: str, temperature: float, unit: str = "celsius"):
    print(f"Setting weather for {city} to {temperature}° {unit}")
    return {"status": "success", "city": city, "temperature": temperature}
"""

'\ndef set_weather(city: str, temperature: float, unit: str = "celsius"):\n    print(f"Setting weather for {city} to {temperature}° {unit}")\n    return {"status": "success", "city": city, "temperature": temperature}\n'

Now let's write a description for it.

What did you write?

Optionally, you can test it after adding this function.

In [56]:
set_weather_tool = {
    "type": "function",
    "name": "set_weather",
    "description": "Sets the current weather for a specific city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "The city, e.g., San Francisco"},
            "temp": {"type": "number", "description": "The temperature value."},
        },
        "required": ["city", "temp"],
        "additionalProperties": False
    }
}

In [57]:
tools.add_tool(set_weather, set_weather_tool)

In [58]:
tools.get_tools()

[{'type': 'function',
  'name': 'get_weather',
  'description': 'Get weather function',
  'parameters': {'type': 'object',
   'properties': {'city': {'type': 'string',
     'description': 'Provide city text to look up weather in that city'}},
   'required': ['city'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'set_weather',
  'description': 'Sets the current weather for a specific city.',
  'parameters': {'type': 'object',
   'properties': {'city': {'type': 'string',
     'description': 'The city, e.g., San Francisco'},
    'temp': {'type': 'number', 'description': 'The temperature value.'}},
   'required': ['city', 'temp'],
   'additionalProperties': False}}]

In [59]:
developer_prompt = """
You're a weather assistant. 
You're given a question about weather in some city and your task is to answer it.

If you don't have answer for specific city please ask to set up weather to the city.
Also you can assist to set weather to some city.
""".strip()

chat_interface = chat_assistant.ChatInterface()

chat = chat_assistant.ChatAssistant(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    client=client
)

In [60]:
chat.run()

Chat ended.



## MCP

MCP stands for Model-Context Protocol. It allows LLMs communicate with different tools (like Qdrant). It's function calling, but one step further:
- A tool can export a list of functions it has
- When we include the tool to our Agent, we just need to include the link to the MCP server


### Q3. Install FastMCP
Let's install a library for MCP - FastMCP:

In [61]:
!pip install fastmcp

2121.78s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 358.5 kB/s eta 0:00:00a 0:00:01
  Using cached pydantic-2.11.7-py3-none-any.whl.metadata (67 kB)
  Preparing metadata (setup.py) ... done
  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
  Using cached docstring_parser-0.16-py3-none-any.whl.metadata (3.0 kB)
  Using cached anyio-4.9.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached jsonschema-4.24.0-py3-none-any.whl.metadata (7.8 kB)
  Using cached uvicorn-0.35.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached pydantic_core-2.33.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached typing_inspection-0.4.1-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 kB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 

In [62]:
# fastmcp-2.10.5

### Q4. Simple MCP Server
A simple MCP server from the documentation looks like that:

```python
# weather_server.py
from fastmcp import FastMCP

mcp = FastMCP("Demo 🚀")

@mcp.tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

if __name__ == "__main__":
    mcp.run()
```

In [65]:
def get_weather(city: str) -> float:
    """
    Retrieves the temperature for a specified city.

    Parameters:
        city (str): The name of the city for which to retrieve weather data.

    Returns:
        float: The temperature associated with the city.
    """
    city = city.strip().lower()

    if city in known_weather_data:
        return known_weather_data[city]

    return round(random.uniform(-5, 35), 1)


def set_weather(city: str, temp: float) -> None:
    """
    Sets the temperature for a specified city.

    Parameters:
        city (str): The name of the city for which to set the weather data.
        temp (float): The temperature to associate with the city.

    Returns:
        str: A confirmation string 'OK' indicating successful update.
    """
    city = city.strip().lower()
    known_weather_data[city] = temp
    return 'OK'

Starting MCP server "Starting MCP server 'Demo 🚀' with server.py:1371 transport '<TODO>'" with transport 'stdio'

### Q5. Protocol

There are different ways to communicate with an MCP server. Ours is currently running using standart input/output, which means that the client write something to stdin and read the answer using stdout.

Our weather server is currently running.

This is how we start communitcating with it:
- First, we send an initialization request -- this way, we register our client with the server:

```json
{"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2024-11-05", "capabilities": {"roots": {"listChanged": true}, "sampling": {}}, "clientInfo": {"name": "test-client", "version": "1.0.0"}}}
```

We should get back something like that, which is an aknowledgement of the request:

```json
{"jsonrpc":"2.0","id":1,"result":{"protocolVersion":"2024-11-05","capabilities":{"experimental":{},"prompts":{"listChanged":false},"resources":{"subscribe":false,"listChanged":false},"tools":{"listChanged":true}},"serverInfo":{"name":"Demo 🚀","version":"1.9.4"}}}
```

- Next, we reply back, confirming the initialization:

```json
{"jsonrpc": "2.0", "method": "notifications/initialized"}
```

We don't expect to get anything in response
- Now we can ask for a list of available methods:

```json
{"jsonrpc": "2.0", "id": 2, "method": "tools/list"}
```

Response
```json
{"jsonrpc":"2.0","id":2,"result":
{"tools":[{"name":"get_weather","description":"Retrieves the temperature for a specified city.\n\nParameters:\n    city (str): The name of the city for which to retrieve weather data.\n\nReturns:\n    float: The temperature associated with the city.","inputSchema":{"properties":{"city":{"title":"City","type":"string"}},"required":["city"],"type":"object"},"outputSchema":{"properties":{"result":{"title":"Result","type":"number"}},"required":["result"],"title":"_WrappedResult","type":"object","x-fastmcp-wrap-result":true}},
{"name":"set_weather","description":"Sets the temperature for a specified city.\n\nParameters:\n    city (str): The name of the city for which to set the weather data.\n    temp (float): The temperature to associate with the city.\n\nReturns:\n    str: A confirmation string 'OK' indicating successful update.","inputSchema":{"properties":{"city":{"title":"City","type":"string"},"temp":{"title":"Temp","type":"number"}},"required":["city","temp"],"type":"object"}}]}}
```

- Let's ask the temperature in Berlin:

```json
{"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "<TODO>", "arguments": {<TODO>}}}
```

*Request:*
```json 
{"jsonrpc": "2.0","id": 3,"method": "tools/call","params": {"name": "get_weather","arguments": {"city": "Berlin"}}}
```
*Response:*
```json
{"jsonrpc":"2.0","id":3,"result":{"content":[{"type":"text","text":"20.0"}],"structuredContent":{"result":20.0},"isError":false}}
```

- What did you get in response?

### Q6. Client

We typically don't interact with the server by copy-pasting commands in the terminal.

In practice, we use an MCP Client. Let's implement it.

FastMCP also supports MCP clients:
```py
from fastmcp import Client

async def main():
    async with Client(<TODO>) as mcp_client:
        # TODO
```
Use the client to get the list of available tools of our script. How does the result look like?

If you're running this code in Jupyter, you need to pass an instance of MCP server to the Client:
```py
import weather_server

async def main():
    async with Client(weather_server.mcp) as mcp_client:
        # ....
```
If you run it in a script, you will need to use asyncio:
```py
import asyncio

async def main():
    async with Client("weather_server.py") as mcp_client:
        # ...

if __name__ == "__main__":
    test = asyncio.run(main())
```
Copy the output with the available tools when filling in the homework form.